In [1]:

# RGIA Airport Operations & Traffic Analytics
# Notebook 01: Data Acquisition & Audit

PROJECT_NAME = "RGIA Airport Operations & Traffic Analytics"
PRIMARY_SOURCE = "Airports Authority of India (AAI)"
SOURCE_PAGE = "https://www.aai.aero/en/business-opportunities/aai-traffic-news"

print(PROJECT_NAME)
print(f"Primary Source: {PRIMARY_SOURCE}")
print(f"Source Page: {SOURCE_PAGE}")

RGIA Airport Operations & Traffic Analytics
Primary Source: Airports Authority of India (AAI)
Source Page: https://www.aai.aero/en/business-opportunities/aai-traffic-news


In [2]:
!pip -q install requests beautifulsoup4 pandas lxml openpyxl pdfplumber

import requests
import pandas as pd
import numpy as np
import re
import os
from bs4 import BeautifulSoup
from io import BytesIO

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 102.4 MB/s eta 0:00:00
Libraries loaded successfully.


In [4]:
AAI_URL = "https://www.aai.aero/en/business-opportunities/aai-traffic-news"

response = requests.get(
    AAI_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status code:", response.status_code)
print("Page size:", len(response.content), "bytes")

Status code: 200
Page size: 930164 bytes


In [5]:
soup = BeautifulSoup(response.content, "html.parser")

links = soup.find_all("a")

aai_links = []

for link in links:
    href = link.get("href")
    text = link.get_text(" ", strip=True)

    if href and (
        "Annexure" in text
        or "Traffic News" in text
        or "Passengers" in text
        or "Freight" in text
        or "Aircraft Movement" in text
    ):
        aai_links.append({
            "text": text,
            "url": href
        })

aai_links[:20]

[{'text': 'Traffic News',
  'url': '/en/business-opportunities/aai-traffic-news'},
 {'text': 'Traffic News',
  'url': '/en/business-opportunities/aai-traffic-news'},
 {'text': 'Traffic News Summary',
  'url': 'https://www.aai.aero/sites/default/files/traffic-news/TRAug2k26.pdf'},
 {'text': 'Traffic News Summary',
  'url': '/en/node/747348?width=700px&height=500px&inline=true#colorbox-inline-106330072'},
 {'text': 'Traffic News Summary',
  'url': 'https://www.aai.aero/sites/default/files/traffic-news/TRAug2k26.pdf'},
 {'text': 'Annexure-I-Airport Category wise Traffic Summary',
  'url': 'https://www.aai.aero/sites/default/files/traffic-news/Aug2k26Annex1.pdf'},
 {'text': 'Annexure-I-Airport Category wise Traffic Summary',
  'url': '/en/node/747349?width=700px&height=500px&inline=true#colorbox-inline-953443811'},
 {'text': 'Annexure-I-Airport Category wise Traffic Summary',
  'url': 'https://www.aai.aero/sites/default/files/traffic-news/Aug2k26Annex1.pdf'},
 {'text': 'Annexure-II-Aircraf

In [7]:

#August 2026 Passenger Report

PASSENGER_URL = (
    "https://www.aai.aero/sites/default/files/"
    "traffic-news/Aug2k26Annex3.pdf"
)

pdf_response = requests.get(
    PASSENGER_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status code:", pdf_response.status_code)
print("File size:", round(len(pdf_response.content) / 1024, 2), "KB")
print("Content type:", pdf_response.headers.get("Content-Type"))

Status code: 200
File size: 488.7 KB
Content type: application/pdf


In [8]:
import pdfplumber

with pdfplumber.open(BytesIO(pdf_response.content)) as pdf:

    print("Number of pages:", len(pdf.pages))

    for i, page in enumerate(pdf.pages[:3]):
        text = page.extract_text()

        print(f"\n{'='*70}")
        print(f"PAGE {i + 1}")
        print(f"{'='*70}")

        if text:
            print(text[:5000])
        else:
            print("No text extracted.")

Number of pages: 7

PAGE 1
अनुलग्नक-III क ANNEXURE-III A
अंर्िरतष्ट्रीय यरत्री International Passengers
यरत्री (संख्यर में ) PASSENGERS (IN NOS.)
मरह के लिए For The Month अवलि के लिए For The Period
अगस्त अगस्त अप्रैि से अगस्त
क्र.सं % %
एयिपोर्त Airport August August April To August
S.NO
परिवर्तन परिवर्तन
2026 2025 2026-2027 2025-2026
Change Change
अंर्िरतष्ट्रीय एयिपोर्तस INTERNATIONAL AIRPORTS
1 अमृतसर AMRITSAR 65197 80663 -19.2% 298064 400206 -25.5%
2 अयोध्या AYODHYA 0 0 - 0 0 -
3 भुवनेश्वर BHUBANESWAR 3 10384 - 35207 46984 -25.1%
4 चेन्नई CHENNAI 488949 527736 -7.3% 2414765 2621778 -7.9%
5 कोयंबत्तूर COIMBATORE 22722 28990 -21.6% 99276 136597 -27.3%
6 गोवा GOA (DABOLIM) 6372 14822 -57.0% 24647 66237 -62.8%
7 इंफाल IMPHAL 0 0 - 0 0 -
8 कोलकाता KOLKATA 158624 172088 -7.8% 930440 925467 0.5%
9 कु शीनगर KUSHINAGAR 0 0 - 0 0 -
10 कोझिकोड KOZHIKODE 260932 285875 -8.7% 1045930 1333715 -21.6%
11 मदुरै MADURAI 9677 18258 -47.0% 31811 84446 -62.3%
12 पोर्ट ब्लेयर PORT BLAIR 0 0 - 0 1017 -
13

In [9]:

#August 2026 Aircraft Movement Report


MOVEMENT_URL = (
    "https://www.aai.aero/sites/default/files/"
    "traffic-news/Aug2k26Annex2.pdf"
)

movement_response = requests.get(
    MOVEMENT_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status code:", movement_response.status_code)
print("File size:", round(len(movement_response.content) / 1024, 2), "KB")
print("Content type:", movement_response.headers.get("Content-Type"))

Status code: 200
File size: 490.25 KB
Content type: application/pdf


In [10]:

with pdfplumber.open(BytesIO(movement_response.content)) as pdf:

    print("Number of pages:", len(pdf.pages))

    for i, page in enumerate(pdf.pages[:3]):
        text = page.extract_text()

        print(f"\n{'='*70}")
        print(f"PAGE {i + 1}")
        print(f"{'='*70}")

        if text:
            print(text[:5000])
        else:
            print("No text extracted.")

Number of pages: 7

PAGE 1
अनुलग्नक-II क ANNEXURE-II A
अंर्िातष्ट्रीय वायुयान प्रचालन International Aircraft Movement
वायुयान प्रचालन (संख्या में ) AIRCRAFT MOVEMENTS ( IN NOS)
माह के नलए For The Month अवनध के नलए For The Period
अगस्त अगस्त अप्रैल से अगस्त
क्र.सं % %
एयिपोर्त Airport August August April To August
S.NO
परिवर्तन परिवर्तन
2026 2025 2026-2027 2025-2026
Change Change
अंर्िातष्ट्रीय एयिपोर्तस INTERNATIONAL AIRPORTS
1 अमृतसर AMRITSAR 401 472 -15.0% 1891 2437 -22.4%
2 अयोध्या AYODHYA 0 0 - 0 0 -
3 भुवनेश्वर BHUBANESWAR 3 93 -96.8% 280 386 -27.5%
4 चेन्नई CHENNAI 3302 3593 -8.1% 16228 17671 -8.2%
5 कोयंबत्तूर COIMBATORE 140 185 -24.3% 579 882 -34.4%
6 इंफाल IMPHAL 0 0 - 0 0 -
7 गोवा GOA (DABOLIM) 62 112 -44.6% 250 530 -52.8%
8 कोलकाता KOLKATA 1434 1388 3.3% 7681 7383 4.0%
9 कोझिकोड KOZHIKODE 1669 1746 -4.4% 6729 8410 -20.0%
10 कु शीनगर KUSHINAGAR 0 0 - 0 0 -
11 मदुरै MADURAI 76 139 -45.3% 244 628 -61.1%
12 पोर्ट ब्लेयर PORT BLAIR 0 0 - 0 10 -
13 राजकोर् (हीरासर) RAJKOT (HIRASAR

In [11]:
FREIGHT_URL = (
    "https://www.aai.aero/sites/default/files/"
    "traffic-news/Aug2k26Annex_4.pdf"
)

freight_response = requests.get(
    FREIGHT_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status code:", freight_response.status_code)
print("File size:", round(len(freight_response.content) / 1024, 2), "KB")
print("Content type:", freight_response.headers.get("Content-Type"))

Status code: 200
File size: 128.23 KB
Content type: application/pdf


In [12]:
with pdfplumber.open(BytesIO(freight_response.content)) as pdf:

    print("Number of pages:", len(pdf.pages))

    for i, page in enumerate(pdf.pages[:3]):
        text = page.extract_text()

        print(f"\n{'='*70}")
        print(f"PAGE {i + 1}")
        print(f"{'='*70}")

        if text:
            print(text[:5000])
        else:
            print("No text extracted.")

Number of pages: 7

PAGE 1
अनलु @नक-IV क ANNEXURE-IV A
अंतरा(cid:9)(cid:25)(cid:26)(cid:27)य माल International Freight
माल (म&ै(cid:26)क टन म') Freight (In MT )
माह के *लए For The Month अव+ध के *लए For The Period
(.स ं
एयरपोट(cid:9) Airport A
अ
u
ग
g
(cid:3)
u
त
s t A
अ
u
ग
g
(cid:3)
u
त
s t
%
Ap
अ
r
,
il
ैल
T
स
o
े अ
A
ग
u
(cid:3)
g
त
u st
%
S.NO
2026 2025 प(cid:6)रवतन(cid:9) Change 2026-2027 2025-2026
प(cid:6)रवतन(cid:9)
Change
अंतरा(cid:9)(cid:25)(cid:26)(cid:27)य एयरपोट(cid:9)स INTERNATIONAL AIRPORTS
1अमतृ सर AMRITSAR 187.6 158.3 18.5% 885.8 760.5 16.5%
2अयो(cid:9)या AYODHYA 0.0 0.0 - 0.0 0.0 -
3भवु ने(cid:16)वर BHUBANESWAR 0.0 5.6 - 0.0 49.0 -
4चे(cid:18)नई CHENNAI 33955.1 27609.9 23.0% 158280.3 139395.6 13.5%
5कोयबं (cid:23)रू COIMBATORE 140.9 172.3 -18.2% 756.2 924.1 -18.2%
6गोवा GOA (DABOLIM) 2.3 47.6 -95.3% 28.6 138.9 -79.4%
7इंफाल IMPHAL 0.0 0.0 - 0.0 0.0 -
8कोलकाता KOLKATA 4713.7 4947.5 -4.7% 21527.2 20824.8 3.4%
9को(cid:29)झकोड KOZHIKODE 1505.4 1700.4 -11.5% 6312.4 8361.9 -

In [13]:
def extract_pdf_text(pdf_bytes):
    """
    Extract text from all pages of an AAI PDF.
    Returns a list containing page-level text.
    """

    pages_text = []

    with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            pages_text.append(text if text else "")

    return pages_text


passenger_pages = extract_pdf_text(pdf_response.content)
movement_pages = extract_pdf_text(movement_response.content)
freight_pages = extract_pdf_text(freight_response.content)

print("Passenger pages:", len(passenger_pages))
print("Movement pages:", len(movement_pages))
print("Freight pages:", len(freight_pages))

Passenger pages: 7
Movement pages: 7
Freight pages: 7


In [14]:
def find_rgya_lines(pages_text):
    """
    Find lines containing Hyderabad (GHIAL)
    or Hyderabad GHIAL references.
    """

    matches = []

    for page_no, page_text in enumerate(pages_text, start=1):
        for line in page_text.splitlines():
            if "HYDERABAD (GHIAL)" in line.upper():
                matches.append({
                    "page": page_no,
                    "line": line
                })

    return matches


print("PASSENGER RGIA:")
for x in find_rgya_lines(passenger_pages):
    print(x)

print("\nMOVEMENT RGIA:")
for x in find_rgya_lines(movement_pages):
    print(x)

print("\nFREIGHT RGIA:")
for x in find_rgya_lines(freight_pages):
    print(x)

PASSENGER RGIA:
{'page': 1, 'line': '3 हैदराबाद HYDERABAD (GHIAL) 452164 464341 -2.6% 2117531 2261073 -6.3%'}
{'page': 2, 'line': '3 हैदराबाद HYDERABAD (GHIAL) 1733770 2005921 -13.6% 9379416 10734561 -12.6%'}
{'page': 5, 'line': '3 हैदराबाद HYDERABAD (GHIAL) 2185934 2470262 -11.5% 11496947 12995634 -11.5%'}

MOVEMENT RGIA:
{'page': 1, 'line': '3 हैदराबाद HYDERABAD (GHIAL) 2730 2834 -3.7% 12504 13545 -7.7%'}
{'page': 2, 'line': '3 हैदराबाद HYDERABAD (GHIAL) 12082 14411 -16.2% 65163 76564 -14.9%'}
{'page': 5, 'line': '3 हैदराबाद HYDERABAD (GHIAL) 14812 17245 -14.1% 77667 90109 -13.8%'}

FREIGHT RGIA:
{'page': 1, 'line': '3हैदराबाद HYDERABAD (GHIAL) 11640.6 8725.1 33.4% 56520.2 41546.7 36.0%'}
{'page': 2, 'line': '3हैदराबाद HYDERABAD (GHIAL) 5848.5 6232.5 -6.2% 29719.7 30580.9 -2.8%'}
{'page': 5, 'line': '3हैदराबाद HYDERABAD (GHIAL) 17489.1 14957.6 16.9% 86240.0 72127.6 19.6%'}


In [15]:
import re
import pandas as pd


def parse_rgya_line(line):
    """
    Extract the six numeric fields from an RGIA HYDERABAD (GHIAL) row.

    Expected structure:
    Airport
    current_month
    previous_year_month
    monthly_change
    current_period
    previous_year_period
    period_change
    """

    # Remove serial number and airport name
    line_clean = re.sub(
        r"^\s*\d+\s*.*?HYDERABAD\s*\(GHIAL\)\s*",
        "",
        line,
        flags=re.IGNORECASE
    )

    # Extract numeric values
    values = re.findall(
        r"-?\d+(?:\.\d+)?%?",
        line_clean
    )

    if len(values) != 6:
        return None

    return {
        "month_current": float(values[0].replace("%", "")),
        "month_previous_year": float(values[1].replace("%", "")),
        "month_change_pct": float(values[2].replace("%", "")),
        "period_current": float(values[3].replace("%", "")),
        "period_previous_year": float(values[4].replace("%", "")),
        "period_change_pct": float(values[5].replace("%", ""))
    }

In [16]:
test_lines = [
    "3 हैदराबाद HYDERABAD (GHIAL) 452164 464341 -2.6% 2117531 2261073 -6.3%",
    "3 हैदराबाद HYDERABAD (GHIAL) 1733770 2005921 -13.6% 9379416 10734561 -12.6%",
    "3 हैदराबाद HYDERABAD (GHIAL) 2185934 2470262 -11.5% 11496947 12995634 -11.5%",

    "3 हैदराबाद HYDERABAD (GHIAL) 2730 2834 -3.7% 12504 13545 -7.7%",
    "3 हैदराबाद HYDERABAD (GHIAL) 12082 14411 -16.2% 65163 76564 -14.9%",
    "3 हैदराबाद HYDERABAD (GHIAL) 14812 17245 -14.1% 77667 90109 -13.8%",

    "3हैदराबाद HYDERABAD (GHIAL) 11640.6 8725.1 33.4% 56520.2 41546.7 36.0%",
    "3हैदराबाद HYDERABAD (GHIAL) 5848.5 6232.5 -6.2% 29719.7 30580.9 -2.8%",
    "3हैदराबाद HYDERABAD (GHIAL) 17489.1 14957.6 16.9% 86240.0 72127.6 19.6%"
]

for line in test_lines:
    print(parse_rgya_line(line))

{'month_current': 452164.0, 'month_previous_year': 464341.0, 'month_change_pct': -2.6, 'period_current': 2117531.0, 'period_previous_year': 2261073.0, 'period_change_pct': -6.3}
{'month_current': 1733770.0, 'month_previous_year': 2005921.0, 'month_change_pct': -13.6, 'period_current': 9379416.0, 'period_previous_year': 10734561.0, 'period_change_pct': -12.6}
{'month_current': 2185934.0, 'month_previous_year': 2470262.0, 'month_change_pct': -11.5, 'period_current': 11496947.0, 'period_previous_year': 12995634.0, 'period_change_pct': -11.5}
{'month_current': 2730.0, 'month_previous_year': 2834.0, 'month_change_pct': -3.7, 'period_current': 12504.0, 'period_previous_year': 13545.0, 'period_change_pct': -7.7}
{'month_current': 12082.0, 'month_previous_year': 14411.0, 'month_change_pct': -16.2, 'period_current': 65163.0, 'period_previous_year': 76564.0, 'period_change_pct': -14.9}
{'month_current': 14812.0, 'month_previous_year': 17245.0, 'month_change_pct': -14.1, 'period_current': 77667.0

In [17]:
for name, pages in [
    ("Passengers", passenger_pages),
    ("Aircraft Movements", movement_pages),
    ("Freight", freight_pages)
]:

    print(f"\n{'='*70}")
    print(name.upper())
    print(f"{'='*70}")

    for page_no in [1, 2, 5]:
        print(f"\nPAGE {page_no}")

        for line in pages[page_no - 1].splitlines():
            if "HYDERABAD" in line.upper() and "GHIAL" in line.upper():
                print(line)


PASSENGERS

PAGE 1
3 हैदराबाद HYDERABAD (GHIAL) 452164 464341 -2.6% 2117531 2261073 -6.3%

PAGE 2
3 हैदराबाद HYDERABAD (GHIAL) 1733770 2005921 -13.6% 9379416 10734561 -12.6%

PAGE 5
3 हैदराबाद HYDERABAD (GHIAL) 2185934 2470262 -11.5% 11496947 12995634 -11.5%

AIRCRAFT MOVEMENTS

PAGE 1
3 हैदराबाद HYDERABAD (GHIAL) 2730 2834 -3.7% 12504 13545 -7.7%

PAGE 2
3 हैदराबाद HYDERABAD (GHIAL) 12082 14411 -16.2% 65163 76564 -14.9%

PAGE 5
3 हैदराबाद HYDERABAD (GHIAL) 14812 17245 -14.1% 77667 90109 -13.8%

FREIGHT

PAGE 1
3हैदराबाद HYDERABAD (GHIAL) 11640.6 8725.1 33.4% 56520.2 41546.7 36.0%

PAGE 2
3हैदराबाद HYDERABAD (GHIAL) 5848.5 6232.5 -6.2% 29719.7 30580.9 -2.8%

PAGE 5
3हैदराबाद HYDERABAD (GHIAL) 17489.1 14957.6 16.9% 86240.0 72127.6 19.6%


In [18]:
def parse_rgya_line(line):
    """
    Parse an AAI RGIA row containing:

    month_current
    month_previous_year
    month_change_pct
    period_current
    period_previous_year
    period_change_pct
    """

    # Find the part after HYDERABAD (GHIAL)
    match = re.search(
        r"HYDERABAD\s*\(GHIAL\)\s*(.*)$",
        line,
        flags=re.IGNORECASE
    )

    if not match:
        return None

    values = match.group(1).strip().split()

    if len(values) < 6:
        return None

    def parse_number(value):
        value = value.replace(",", "")
        return float(value)

    def parse_pct(value):
        value = value.replace("%", "")

        if value == "-":
            return np.nan

        return float(value)

    return {
        "month_current": parse_number(values[0]),
        "month_previous_year": parse_number(values[1]),
        "month_change_pct": parse_pct(values[2]),
        "period_current": parse_number(values[3]),
        "period_previous_year": parse_number(values[4]),
        "period_change_pct": parse_pct(values[5])
    }

In [19]:
parse_rgya_line(
    "3 हैदराबाद HYDERABAD (GHIAL) "
    "452164 464341 -2.6% "
    "2117531 2261073 -6.3%"
)


{'month_current': 452164.0,
 'month_previous_year': 464341.0,
 'month_change_pct': -2.6,
 'period_current': 2117531.0,
 'period_previous_year': 2261073.0,
 'period_change_pct': -6.3}

In [20]:
def extract_rgia_metric(pdf_bytes, metric_name, report_month):
    """
    Extract International, Domestic, and Combined RGIA records
    from an AAI traffic annexure PDF.
    """

    pages = extract_pdf_text(pdf_bytes)

    traffic_type_by_page = {
        1: "International",
        2: "Domestic",
        5: "Combined"
    }

    rows = []

    for page_no, traffic_type in traffic_type_by_page.items():

        page_text = pages[page_no - 1]

        for line in page_text.splitlines():

            if "HYDERABAD" in line.upper() and "GHIAL" in line.upper():

                parsed = parse_rgya_line(line)

                if parsed is not None:

                    rows.append({
                        "report_month": report_month,
                        "airport": "HYDERABAD (GHIAL)",
                        "metric": metric_name,
                        "traffic_type": traffic_type,
                        **parsed
                    })

    return pd.DataFrame(rows)

In [21]:
passenger_df = extract_rgia_metric(
    pdf_response.content,
    "Passengers",
    "2026-08"
)

movement_df = extract_rgia_metric(
    movement_response.content,
    "Aircraft Movements",
    "2026-08"
)

freight_df = extract_rgia_metric(
    freight_response.content,
    "Freight",
    "2026-08"
)

august_rgia = pd.concat(
    [passenger_df, movement_df, freight_df],
    ignore_index=True
)

august_rgia

,report_month,airport,metric,traffic_type,month_current,month_previous_year,month_change_pct,period_current,period_previous_year,period_change_pct
0,2026-08,HYDERABAD (GHIAL),Passengers,International,452164.0,464341.0,-2.6,2117531.0,2261073.0,-6.3
1,2026-08,HYDERABAD (GHIAL),Passengers,Domestic,1733770.0,2005921.0,-13.6,9379416.0,10734561.0,-12.6
2,2026-08,HYDERABAD (GHIAL),Passengers,Combined,2185934.0,2470262.0,-11.5,11496947.0,12995634.0,-11.5
3,2026-08,HYDERABAD (GHIAL),Aircraft Movements,International,2730.0,2834.0,-3.7,12504.0,13545.0,-7.7
4,2026-08,HYDERABAD (GHIAL),Aircraft Movements,Domestic,12082.0,14411.0,-16.2,65163.0,76564.0,-14.9
5,2026-08,HYDERABAD (GHIAL),Aircraft Movements,Combined,14812.0,17245.0,-14.1,77667.0,90109.0,-13.8
6,2026-08,HYDERABAD (GHIAL),Freight,International,11640.6,8725.1,33.4,56520.2,41546.7,36.0
7,2026-08,HYDERABAD (GHIAL),Freight,Domestic,5848.5,6232.5,-6.2,29719.7,30580.9,-2.8
8,2026-08,HYDERABAD (GHIAL),Freight,Combined,17489.1,14957.6,16.9,86240.0,72127.6,19.6


In [22]:
print("Rows:", len(august_rgia))
print("Columns:", len(august_rgia.columns))

print("\nMetrics:")
print(august_rgia["metric"].value_counts())

print("\nTraffic types:")
print(august_rgia["traffic_type"].value_counts())

print("\nDuplicates:")
print(
    august_rgia.duplicated(
        subset=["report_month", "metric", "traffic_type"]
    ).sum()
)

Rows: 9
Columns: 10

Metrics:
metric
Passengers            3
Aircraft Movements    3
Freight               3
Name: count, dtype: int64

Traffic types:
traffic_type
International    3
Domestic         3
Combined         3
Name: count, dtype: int64

Duplicates:
0


In [23]:
def validate_rgia_month(df):
    """
    Validate RGIA monthly totals against
    International + Domestic components.
    """

    results = []

    for metric in df["metric"].unique():

        metric_df = df[df["metric"] == metric]

        international = metric_df.loc[
            metric_df["traffic_type"] == "International",
            "month_current"
        ].iloc[0]

        domestic = metric_df.loc[
            metric_df["traffic_type"] == "Domestic",
            "month_current"
        ].iloc[0]

        combined = metric_df.loc[
            metric_df["traffic_type"] == "Combined",
            "month_current"
        ].iloc[0]

        calculated_total = international + domestic
        difference = combined - calculated_total

        results.append({
            "metric": metric,
            "international": international,
            "domestic": domestic,
            "reported_combined": combined,
            "calculated_combined": calculated_total,
            "difference": difference
        })

    return pd.DataFrame(results)


reconciliation = validate_rgia_month(august_rgia)

reconciliation

,metric,international,domestic,reported_combined,calculated_combined,difference
0,Passengers,452164.0,1733770.0,2185934.0,2185934.0,0.0
1,Aircraft Movements,2730.0,12082.0,14812.0,14812.0,0.0
2,Freight,11640.6,5848.5,17489.1,17489.1,0.0


In [24]:
def calculate_yoy(current, previous):
    if previous == 0:
        return np.nan

    return ((current - previous) / previous) * 100


august_rgia["calculated_month_change_pct"] = august_rgia.apply(
    lambda row: calculate_yoy(
        row["month_current"],
        row["month_previous_year"]
    ),
    axis=1
)

august_rgia["pct_difference"] = (
    august_rgia["calculated_month_change_pct"]
    - august_rgia["month_change_pct"]
)

august_rgia[
    [
        "metric",
        "traffic_type",
        "month_change_pct",
        "calculated_month_change_pct",
        "pct_difference"
    ]
]

,metric,traffic_type,month_change_pct,calculated_month_change_pct,pct_difference
0,Passengers,International,-2.6,-2.622426,-0.022426
1,Passengers,Domestic,-13.6,-13.567384,0.032616
2,Passengers,Combined,-11.5,-11.510034,-0.010034
3,Aircraft Movements,International,-3.7,-3.669725,0.030275
4,Aircraft Movements,Domestic,-16.2,-16.161266,0.038734
5,Aircraft Movements,Combined,-14.1,-14.108437,-0.008437
6,Freight,International,33.4,33.415090,0.015090
7,Freight,Domestic,-6.2,-6.161252,0.038748
8,Freight,Combined,16.9,16.924507,0.024507


In [25]:
# Inspecting AAI traffic-report PDF links

pdf_links = []

for link in soup.find_all("a"):
    href = link.get("href")
    text = link.get_text(" ", strip=True)

    if href and ".pdf" in href.lower():
        if href.startswith("/"):
            href = "https://www.aai.aero" + href

        pdf_links.append({
            "text": text,
            "url": href
        })

pdf_links_df = pd.DataFrame(pdf_links)

print("Total PDF links found:", len(pdf_links_df))

pdf_links_df.head(30)

Total PDF links found: 114


,text,url
0,Corporate Plan,https://www.aai.aero/sites/default/files/AAI_C...
1,Corporate Plan,https://www.aai.aero/sites/default/files/AAI_C...
2,Traffic News Summary,https://www.aai.aero/sites/default/files/traff...
3,Traffic News Summary,https://www.aai.aero/sites/default/files/traff...
4,Annexure-I-Airport Category wise Traffic Summary,https://www.aai.aero/sites/default/files/traff...
5,Annexure-I-Airport Category wise Traffic Summary,https://www.aai.aero/sites/default/files/traff...
6,Annexure-II-Aircraft Movement,https://www.aai.aero/sites/default/files/traff...
7,Annexure-II-Aircraft Movement,https://www.aai.aero/sites/default/files/traff...
8,Annexure-III-Passengers,https://www.aai.aero/sites/default/files/traff...
9,Annexure-III-Passengers,https://www.aai.aero/sites/default/files/traff...


In [26]:
# Show PDF filenames so we can inspect AAI's naming pattern

pdf_links_df["filename"] = pdf_links_df["url"].str.split("/").str[-1]

pdf_links_df[
    ["text", "filename", "url"]
].drop_duplicates().head(50)

,text,filename,url
0,Corporate Plan,AAI_CORPORATE_PLAN_2017-26_with_revised_vision...,https://www.aai.aero/sites/default/files/AAI_C...
2,Traffic News Summary,TRAug2k26.pdf,https://www.aai.aero/sites/default/files/traff...
4,Annexure-I-Airport Category wise Traffic Summary,Aug2k26Annex1.pdf,https://www.aai.aero/sites/default/files/traff...
6,Annexure-II-Aircraft Movement,Aug2k26Annex2.pdf,https://www.aai.aero/sites/default/files/traff...
8,Annexure-III-Passengers,Aug2k26Annex3.pdf,https://www.aai.aero/sites/default/files/traff...
10,Annexure-IV-Freight,Aug2k26Annex_4.pdf,https://www.aai.aero/sites/default/files/traff...
12,Annexure-V-Region Wise Trends and Share in Air...,Aug2k26Annex5.pdf,https://www.aai.aero/sites/default/files/traff...
14,Annexure-VI-State/ Union Territory Traffic,Aug2k26Annex6.pdf,https://www.aai.aero/sites/default/files/traff...
16,Traffic News Summary,TRjul2k26.pdf,https://www.aai.aero/sites/default/files/traff...
18,Annexure-I-Airport Category wise Traffic Summary,Jul2k26Annex1.pdf,https://www.aai.aero/sites/default/files/traff...


In [27]:
# Keep only the three annexures required for our project

required_annexures = {
    "Annexure-II-Aircraft Movement": "Aircraft Movements",
    "Annexure-III-Passengers": "Passengers",
    "Annexure-IV-Freight": "Freight"
}

traffic_links = []

for _, row in pdf_links_df.iterrows():

    text = row["text"].strip()

    for source_name, metric_name in required_annexures.items():

        if source_name.lower() in text.lower():

            traffic_links.append({
                "source_text": text,
                "metric": metric_name,
                "url": row["url"],
                "filename": row["filename"]
            })

traffic_links_df = pd.DataFrame(traffic_links)

traffic_links_df = traffic_links_df.drop_duplicates(
    subset=["metric", "url"]
).reset_index(drop=True)

print("Total required PDF links:", len(traffic_links_df))

traffic_links_df.head(30)

Total required PDF links: 24


,source_text,metric,url,filename
0,Annexure-II-Aircraft Movement,Aircraft Movements,https://www.aai.aero/sites/default/files/traff...,Aug2k26Annex2.pdf
1,Annexure-III-Passengers,Passengers,https://www.aai.aero/sites/default/files/traff...,Aug2k26Annex3.pdf
2,Annexure-IV-Freight,Freight,https://www.aai.aero/sites/default/files/traff...,Aug2k26Annex_4.pdf
3,Annexure-II-Aircraft Movement,Aircraft Movements,https://www.aai.aero/sites/default/files/traff...,Jul2k26Annex2.pdf
4,Annexure-III-Passengers,Passengers,https://www.aai.aero/sites/default/files/traff...,Jul2k26Annex3.pdf
5,Annexure-IV-Freight,Freight,https://www.aai.aero/sites/default/files/traff...,Jul2k26Annex4.pdf
6,Annexure-II-Aircraft Movement,Aircraft Movements,https://www.aai.aero/sites/default/files/traff...,Jun2k26Annex2.pdf
7,Annexure-III-Passengers,Passengers,https://www.aai.aero/sites/default/files/traff...,Jun2k26Annex3.pdf
8,Annexure-IV-Freight,Freight,https://www.aai.aero/sites/default/files/traff...,Jun2k26Annex4.pdf
9,Annexure-II-Aircraft Movement,Aircraft Movements,https://www.aai.aero/sites/default/files/traff...,May2k26Annex2.pdf


In [28]:
# Inspect the number of reports available for each metric

traffic_links_df.groupby("metric").size()

,0
metric,
Aircraft Movements,8
Freight,8
Passengers,8


In [29]:
# Display the discovered filenames in chronological-page order

traffic_links_df[
    ["metric", "filename", "url"]
].to_string(index=False)

'            metric            filename                                                                       url\nAircraft Movements   Aug2k26Annex2.pdf   https://www.aai.aero/sites/default/files/traffic-news/Aug2k26Annex2.pdf\n        Passengers   Aug2k26Annex3.pdf   https://www.aai.aero/sites/default/files/traffic-news/Aug2k26Annex3.pdf\n           Freight  Aug2k26Annex_4.pdf  https://www.aai.aero/sites/default/files/traffic-news/Aug2k26Annex_4.pdf\nAircraft Movements   Jul2k26Annex2.pdf   https://www.aai.aero/sites/default/files/traffic-news/Jul2k26Annex2.pdf\n        Passengers   Jul2k26Annex3.pdf   https://www.aai.aero/sites/default/files/traffic-news/Jul2k26Annex3.pdf\n           Freight   Jul2k26Annex4.pdf   https://www.aai.aero/sites/default/files/traffic-news/Jul2k26Annex4.pdf\nAircraft Movements   Jun2k26Annex2.pdf   https://www.aai.aero/sites/default/files/traffic-news/Jun2k26Annex2.pdf\n        Passengers   Jun2k26Annex3.pdf   https://www.aai.aero/sites/default/files/traff

In [30]:

# AAI TRAFFIC REPORT INVENTORY\
inventory = (
    traffic_links_df
    .groupby(["metric"])
    .size()
    .reset_index(name="pdf_count")
)

print("PDF INVENTORY")
print("=" * 50)
print(inventory.to_string(index=False))

print("\n\nUNIQUE PDF URLs:", traffic_links_df["url"].nunique())

print("\n\nFIRST 10 REPORTS")
print("=" * 50)

print(
    traffic_links_df[
        ["metric", "filename"]
    ].head(10).to_string(index=False)
)

print("\n\nLAST 10 REPORTS")
print("=" * 50)

print(
    traffic_links_df[
        ["metric", "filename"]
    ].tail(10).to_string(index=False)
)

PDF INVENTORY
            metric  pdf_count
Aircraft Movements          8
           Freight          8
        Passengers          8


UNIQUE PDF URLs: 24


FIRST 10 REPORTS
            metric           filename
Aircraft Movements  Aug2k26Annex2.pdf
        Passengers  Aug2k26Annex3.pdf
           Freight Aug2k26Annex_4.pdf
Aircraft Movements  Jul2k26Annex2.pdf
        Passengers  Jul2k26Annex3.pdf
           Freight  Jul2k26Annex4.pdf
Aircraft Movements  Jun2k26Annex2.pdf
        Passengers  Jun2k26Annex3.pdf
           Freight  Jun2k26Annex4.pdf
Aircraft Movements  May2k26Annex2.pdf


LAST 10 REPORTS
            metric            filename
           Freight April2k26Annex4.pdf
Aircraft Movements Mar2k26Annex2_0.pdf
        Passengers Mar2k26Annex3_0.pdf
           Freight   Mar2k26Annex4.pdf
Aircraft Movements   Feb2k26Annex2.pdf
        Passengers   Feb2k26Annex3.pdf
           Freight   Feb2k26Annex4.pdf
Aircraft Movements Jan2k26Annex2_0.pdf
        Passengers Jan2k26Annex3_0.pdf

In [32]:
metric_counts = traffic_links_df.groupby("metric").size()

print(metric_counts)

if metric_counts.nunique() == 1:
    print("\n✅ All three metrics have the same number of discovered reports.")
else:
    print("\n⚠️ Metrics have different numbers of reports.")

metric
Aircraft Movements    8
Freight               8
Passengers            8
dtype: int64

✅ All three metrics have the same number of discovered reports.


In [33]:


duplicate_count = traffic_links_df.duplicated(
    subset=["metric", "url"]
).sum()

print("Duplicate metric + URL records:", duplicate_count)

Duplicate metric + URL records: 0


In [34]:


for metric in ["Passengers", "Aircraft Movements", "Freight"]:

    print(f"\n{'=' * 60}")
    print(metric.upper())
    print(f"{'=' * 60}")

    metric_files = traffic_links_df[
        traffic_links_df["metric"] == metric
    ]

    for filename in metric_files["filename"]:
        print(filename)


PASSENGERS
Aug2k26Annex3.pdf
Jul2k26Annex3.pdf
Jun2k26Annex3.pdf
May2K26Annex3.pdf
April2k26Annex3.pdf
Mar2k26Annex3_0.pdf
Feb2k26Annex3.pdf
Jan2k26Annex3_0.pdf

AIRCRAFT MOVEMENTS
Aug2k26Annex2.pdf
Jul2k26Annex2.pdf
Jun2k26Annex2.pdf
May2k26Annex2.pdf
April2k26Annex2.pdf
Mar2k26Annex2_0.pdf
Feb2k26Annex2.pdf
Jan2k26Annex2_0.pdf

FREIGHT
Aug2k26Annex_4.pdf
Jul2k26Annex4.pdf
Jun2k26Annex4.pdf
May2k26Annex4.pdf
April2k26Annex4.pdf
Mar2k26Annex4.pdf
Feb2k26Annex4.pdf
Jan2k26Annex4_0.pdf


In [35]:


print(
    traffic_links_df[
        ["metric", "filename"]
    ].to_string(index=False)
)

            metric            filename
Aircraft Movements   Aug2k26Annex2.pdf
        Passengers   Aug2k26Annex3.pdf
           Freight  Aug2k26Annex_4.pdf
Aircraft Movements   Jul2k26Annex2.pdf
        Passengers   Jul2k26Annex3.pdf
           Freight   Jul2k26Annex4.pdf
Aircraft Movements   Jun2k26Annex2.pdf
        Passengers   Jun2k26Annex3.pdf
           Freight   Jun2k26Annex4.pdf
Aircraft Movements   May2k26Annex2.pdf
        Passengers   May2K26Annex3.pdf
           Freight   May2k26Annex4.pdf
Aircraft Movements April2k26Annex2.pdf
        Passengers April2k26Annex3.pdf
           Freight April2k26Annex4.pdf
Aircraft Movements Mar2k26Annex2_0.pdf
        Passengers Mar2k26Annex3_0.pdf
           Freight   Mar2k26Annex4.pdf
Aircraft Movements   Feb2k26Annex2.pdf
        Passengers   Feb2k26Annex3.pdf
           Freight   Feb2k26Annex4.pdf
Aircraft Movements Jan2k26Annex2_0.pdf
        Passengers Jan2k26Annex3_0.pdf
           Freight Jan2k26Annex4_0.pdf


In [36]:

#AAI REPORT MANIFEST

month_map = {
    "Jan": "2026-01",
    "Feb": "2026-02",
    "Mar": "2026-03",
    "April": "2026-04",
    "May": "2026-05",
    "Jun": "2026-06",
    "Jul": "2026-07",
    "Aug": "2026-08"
}

def identify_report_month(filename):
    filename_lower = filename.lower()

    for month_name, month_value in month_map.items():

        if month_name.lower() in filename_lower:
            return month_value

    return None


manifest = traffic_links_df.copy()

manifest["report_month"] = manifest["filename"].apply(
    identify_report_month
)

manifest = manifest[
    [
        "report_month",
        "metric",
        "filename",
        "url"
    ]
].rename(
    columns={"url": "source_url"}
)

manifest = manifest.sort_values(
    ["report_month", "metric"]
).reset_index(drop=True)

print(manifest.to_string(index=False))

report_month             metric            filename                                                                source_url
     2026-01 Aircraft Movements Jan2k26Annex2_0.pdf https://www.aai.aero/sites/default/files/traffic-news/Jan2k26Annex2_0.pdf
     2026-01            Freight Jan2k26Annex4_0.pdf https://www.aai.aero/sites/default/files/traffic-news/Jan2k26Annex4_0.pdf
     2026-01         Passengers Jan2k26Annex3_0.pdf https://www.aai.aero/sites/default/files/traffic-news/Jan2k26Annex3_0.pdf
     2026-02 Aircraft Movements   Feb2k26Annex2.pdf   https://www.aai.aero/sites/default/files/traffic-news/Feb2k26Annex2.pdf
     2026-02            Freight   Feb2k26Annex4.pdf   https://www.aai.aero/sites/default/files/traffic-news/Feb2k26Annex4.pdf
     2026-02         Passengers   Feb2k26Annex3.pdf   https://www.aai.aero/sites/default/files/traffic-news/Feb2k26Annex3.pdf
     2026-03 Aircraft Movements Mar2k26Annex2_0.pdf https://www.aai.aero/sites/default/files/traffic-news/Mar2k26Annex

In [37]:
# Manifest validation

print("Total records:", len(manifest))
print("Unique months:", manifest["report_month"].nunique())
print("Unique URLs:", manifest["source_url"].nunique())
print("Missing months:", manifest["report_month"].isna().sum())
print("Duplicate records:", manifest.duplicated().sum())

print("\nReports per month:")
print(manifest.groupby("report_month").size())

Total records: 24
Unique months: 8
Unique URLs: 24
Missing months: 0
Duplicate records: 0

Reports per month:
report_month
2026-01    3
2026-02    3
2026-03    3
2026-04    3
2026-05    3
2026-06    3
2026-07    3
2026-08    3
dtype: int64


In [38]:

#PROJECT DATA DIRECTORIES

RAW_DIR = "/content/rgia_airport_analytics/data/raw"
PROCESSED_DIR = "/content/rgia_airport_analytics/data/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

Raw directory: /content/rgia_airport_analytics/data/raw
Processed directory: /content/rgia_airport_analytics/data/processed


In [39]:


download_log = []

for _, row in manifest.iterrows():

    report_month = row["report_month"]
    metric = row["metric"]
    filename = row["filename"]
    url = row["source_url"]

    metric_folder = metric.lower().replace(" ", "_")

    folder = os.path.join(
        RAW_DIR,
        metric_folder
    )

    os.makedirs(folder, exist_ok=True)

    filepath = os.path.join(folder, filename)

    try:

        r = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=60
        )

        r.raise_for_status()

        with open(filepath, "wb") as f:
            f.write(r.content)

        download_log.append({
            "report_month": report_month,
            "metric": metric,
            "filename": filename,
            "status": "Downloaded",
            "size_kb": round(len(r.content) / 1024, 2),
            "filepath": filepath
        })

    except Exception as e:

        download_log.append({
            "report_month": report_month,
            "metric": metric,
            "filename": filename,
            "status": f"ERROR: {e}",
            "size_kb": None,
            "filepath": filepath
        })


download_log_df = pd.DataFrame(download_log)

download_log_df

,report_month,metric,filename,status,size_kb,filepath
0,2026-01,Aircraft Movements,Jan2k26Annex2_0.pdf,Downloaded,1045.75,/content/rgia_airport_analytics/data/raw/aircr...
1,2026-01,Freight,Jan2k26Annex4_0.pdf,Downloaded,1038.77,/content/rgia_airport_analytics/data/raw/freig...
2,2026-01,Passengers,Jan2k26Annex3_0.pdf,Downloaded,1072.56,/content/rgia_airport_analytics/data/raw/passe...
3,2026-02,Aircraft Movements,Feb2k26Annex2.pdf,Downloaded,486.50,/content/rgia_airport_analytics/data/raw/aircr...
4,2026-02,Freight,Feb2k26Annex4.pdf,Downloaded,480.13,/content/rgia_airport_analytics/data/raw/freig...
5,2026-02,Passengers,Feb2k26Annex3.pdf,Downloaded,488.21,/content/rgia_airport_analytics/data/raw/passe...
6,2026-03,Aircraft Movements,Mar2k26Annex2_0.pdf,Downloaded,477.75,/content/rgia_airport_analytics/data/raw/aircr...
7,2026-03,Freight,Mar2k26Annex4.pdf,Downloaded,471.38,/content/rgia_airport_analytics/data/raw/freig...
8,2026-03,Passengers,Mar2k26Annex3_0.pdf,Downloaded,479.68,/content/rgia_airport_analytics/data/raw/passe...
9,2026-04,Aircraft Movements,April2k26Annex2.pdf,Downloaded,479.25,/content/rgia_airport_analytics/data/raw/aircr...


In [40]:
print("Total downloads:", len(download_log_df))

print("\nStatus:")
print(download_log_df["status"].value_counts())

print("\nFailed downloads:")
print(
    download_log_df[
        download_log_df["status"] != "Downloaded"
    ][
        ["report_month", "metric", "filename", "status"]
    ]
)

Total downloads: 24

Status:
status
Downloaded    24
Name: count, dtype: int64

Failed downloads:
Empty DataFrame
Columns: [report_month, metric, filename, status]
Index: []


In [41]:


def inspect_rgia_pages(filepath):
    """
    Find pages containing the RGIA (HYDERABAD / GHIAL)
    record and display the matching lines.
    """

    with open(filepath, "rb") as f:
        pdf_bytes = f.read()

    pages = extract_pdf_text(pdf_bytes)

    results = []

    for page_no, page_text in enumerate(pages, start=1):

        for line in page_text.splitlines():

            if (
                "HYDERABAD" in line.upper()
                and "GHIAL" in line.upper()
            ):
                results.append({
                    "page": page_no,
                    "line": line.strip()
                })

    return pd.DataFrame(results)


# Test one report from each metric
test_files = manifest[
    (manifest["report_month"].isin([
        "2026-01",
        "2026-04",
        "2026-08"
    ]))
].copy()

for _, row in test_files.iterrows():

    metric_folder = row["metric"].lower().replace(" ", "_")

    filepath = os.path.join(
        RAW_DIR,
        metric_folder,
        row["filename"]
    )

    print("=" * 80)
    print("Month :", row["report_month"])
    print("Metric:", row["metric"])
    print("File  :", row["filename"])

    result = inspect_rgia_pages(filepath)

    print(result.to_string(index=False))
    print()

Month : 2026-01
Metric: Aircraft Movements
File  : Jan2k26Annex2_0.pdf
 page                                                              line
    1     3 हैदराबाद HYDERABAD (GHIAL) 2895 2522 14.8% 27535 25115 9.6%
    2 3 हैदराबाद HYDERABAD (GHIAL) 13999 15271 -8.3% 149557 141750 5.5%
    5 3 हैदराबाद HYDERABAD (GHIAL) 16894 17793 -5.1% 177092 166865 6.1%

Month : 2026-01
Metric: Freight
File  : Jan2k26Annex4_0.pdf
 page                                                                      line
    1    3 हैदराबाद HYDERABAD (GHIAL) 9311.5 7302.2 27.5% 90615.5 80941.4 12.0%
    2      3 हैदराबाद HYDERABAD (GHIAL) 5596.8 5487.2 2.0% 60536.5 57730.9 4.9%
    5 3 हैदराबाद HYDERABAD (GHIAL) 14908.4 12789.4 16.6% 151152.0 138672.3 9.0%

Month : 2026-01
Metric: Passengers
File  : Jan2k26Annex3_0.pdf
 page                                                                      line
    1    3 हैदराबाद HYDERABAD (GHIAL) 523439 424013 23.4% 4717724 3935053 19.9%
    2 3 हैदराबाद HYDERABAD (GHIAL) 2

In [42]:
# ============================================================
# Bulk Extraction — RGIA Monthly Traffic Data
# ============================================================

def extract_rgia_from_file(filepath, metric_name, report_month):
    """
    Extract International, Domestic and Combined RGIA records
    from an AAI traffic report PDF.
    """

    with open(filepath, "rb") as f:
        pdf_bytes = f.read()

    pages = extract_pdf_text(pdf_bytes)

    traffic_type_by_page = {
        1: "International",
        2: "Domestic",
        5: "Combined"
    }

    rows = []

    for page_no, traffic_type in traffic_type_by_page.items():

        page_text = pages[page_no - 1]

        rgia_found = False

        for line in page_text.splitlines():

            if (
                "HYDERABAD" in line.upper()
                and "GHIAL" in line.upper()
            ):

                parsed = parse_rgya_line(line)

                if parsed is not None:

                    rows.append({
                        "report_month": report_month,
                        "metric": metric_name,
                        "traffic_type": traffic_type,
                        **parsed
                    })

                    rgia_found = True
                    break

        if not rgia_found:
            print(
                f"WARNING: RGIA row not found | "
                f"{report_month} | {metric_name} | "
                f"Page {page_no}"
            )

    return rows


# ------------------------------------------------------------
# Extract all reports
# ------------------------------------------------------------

all_rows = []

for _, row in manifest.iterrows():

    metric = row["metric"]
    report_month = row["report_month"]
    filename = row["filename"]

    metric_folder = metric.lower().replace(" ", "_")

    filepath = os.path.join(
        RAW_DIR,
        metric_folder,
        filename
    )

    extracted_rows = extract_rgia_from_file(
        filepath,
        metric,
        report_month
    )

    all_rows.extend(extracted_rows)


rgia_traffic_raw = pd.DataFrame(all_rows)

print("Extraction complete.")
print("Rows:", len(rgia_traffic_raw))
print("Columns:", len(rgia_traffic_raw.columns))

rgia_traffic_raw.head(10)

Extraction complete.
Rows: 72
Columns: 9


,report_month,metric,traffic_type,month_current,month_previous_year,month_change_pct,period_current,period_previous_year,period_change_pct
0,2026-01,Aircraft Movements,International,2895.0,2522.0,14.8,27535.0,25115.0,9.6
1,2026-01,Aircraft Movements,Domestic,13999.0,15271.0,-8.3,149557.0,141750.0,5.5
2,2026-01,Aircraft Movements,Combined,16894.0,17793.0,-5.1,177092.0,166865.0,6.1
3,2026-01,Freight,International,9311.5,7302.2,27.5,90615.5,80941.4,12.0
4,2026-01,Freight,Domestic,5596.8,5487.2,2.0,60536.5,57730.9,4.9
5,2026-01,Freight,Combined,14908.4,12789.4,16.6,151152.0,138672.3,9.0
6,2026-01,Passengers,International,523439.0,424013.0,23.4,4717724.0,3935053.0,19.9
7,2026-01,Passengers,Domestic,2129340.0,2218574.0,-4.0,21160774.0,20104882.0,5.3
8,2026-01,Passengers,Combined,2652779.0,2642587.0,0.4,25878498.0,24039935.0,7.6
9,2026-02,Aircraft Movements,International,2573.0,2429.0,5.9,30108.0,27544.0,9.3


In [43]:
# ============================================================
# Validation 1 — International + Domestic = Combined
# ============================================================

reconciliation_rows = []

for (month, metric), group in rgia_traffic_raw.groupby(
    ["report_month", "metric"]
):

    international = group.loc[
        group["traffic_type"] == "International",
        "month_current"
    ].iloc[0]

    domestic = group.loc[
        group["traffic_type"] == "Domestic",
        "month_current"
    ].iloc[0]

    combined = group.loc[
        group["traffic_type"] == "Combined",
        "month_current"
    ].iloc[0]

    calculated_combined = international + domestic

    difference = combined - calculated_combined

    reconciliation_rows.append({
        "report_month": month,
        "metric": metric,
        "international": international,
        "domestic": domestic,
        "reported_combined": combined,
        "calculated_combined": calculated_combined,
        "difference": difference
    })


reconciliation_df = pd.DataFrame(reconciliation_rows)

print("Total reconciliation checks:", len(reconciliation_df))
print(
    "Exact matches:",
    (reconciliation_df["difference"] == 0).sum()
)

print(
    "Maximum absolute difference:",
    reconciliation_df["difference"].abs().max()
)

reconciliation_df

Total reconciliation checks: 24
Exact matches: 21
Maximum absolute difference: 0.10000000000218279


,report_month,metric,international,domestic,reported_combined,calculated_combined,difference
0,2026-01,Aircraft Movements,2895.0,13999.0,16894.0,16894.0,0.000000e+00
1,2026-01,Freight,9311.5,5596.8,14908.4,14908.3,1.000000e-01
2,2026-01,Passengers,523439.0,2129340.0,2652779.0,2652779.0,0.000000e+00
3,2026-02,Aircraft Movements,2573.0,12328.0,14901.0,14901.0,0.000000e+00
4,2026-02,Freight,9992.7,5455.2,15447.9,15447.9,-1.818989e-12
5,2026-02,Passengers,429602.0,1887511.0,2317113.0,2317113.0,0.000000e+00
6,2026-03,Aircraft Movements,1954.0,13953.0,15907.0,15907.0,0.000000e+00
7,2026-03,Freight,9667.2,6172.9,15840.1,15840.1,0.000000e+00
8,2026-03,Passengers,315776.0,1972498.0,2288274.0,2288274.0,0.000000e+00
9,2026-04,Aircraft Movements,2010.0,13819.0,15829.0,15829.0,0.000000e+00


In [ ]:
# ============================================================
# Reconciliation Summary
# ============================================================

reconciliation_df["abs_difference"] = (
    reconciliation_df["difference"].abs()
)

print("Differences by metric:")
print(
    reconciliation_df
    .groupby("metric")["abs_difference"]
    .agg(["max", "mean"])
)

print("\nRecords with non-zero difference:")

print(
    reconciliation_df[
        reconciliation_df["abs_difference"] > 0
    ][
        [
            "report_month",
            "metric",
            "international",
            "domestic",
            "reported_combined",
            "calculated_combined",
            "difference"
        ]
    ]
)

In [46]:
# ============================================================
# Reconciliation Summary
# ============================================================

reconciliation_df["abs_difference"] = (
    reconciliation_df["difference"].abs()
)

print("Differences by metric:")
print(
    reconciliation_df
    .groupby("metric")["abs_difference"]
    .agg(["max", "mean"])
)

print("\nRecords with non-zero difference:")

print(
    reconciliation_df[
        reconciliation_df["abs_difference"] > 0
    ][
        [
            "report_month",
            "metric",
            "international",
            "domestic",
            "reported_combined",
            "calculated_combined",
            "difference"
        ]
    ]
)

Differences by metric:
                    max   mean
metric                        
Aircraft Movements  0.0  0.000
Freight             0.1  0.025
Passengers          0.0  0.000

Records with non-zero difference:
   report_month   metric  international  domestic  reported_combined  \
1       2026-01  Freight         9311.5    5596.8            14908.4   
4       2026-02  Freight         9992.7    5455.2            15447.9   
13      2026-05  Freight        10390.0    6159.4            16549.3   

    calculated_combined    difference  
1               14908.3  1.000000e-01  
4               15447.9 -1.818989e-12  
13              16549.4 -1.000000e-01  


In [45]:
# ============================================================
# Validation 2 — Independent YoY Calculation
# ============================================================

def calculate_yoy(current, previous):

    if previous == 0:
        return np.nan

    return ((current - previous) / previous) * 100


rgia_traffic_raw["calculated_month_change_pct"] = (
    rgia_traffic_raw.apply(
        lambda row: calculate_yoy(
            row["month_current"],
            row["month_previous_year"]
        ),
        axis=1
    )
)

rgia_traffic_raw["yoy_pct_difference"] = (
    rgia_traffic_raw["calculated_month_change_pct"]
    - rgia_traffic_raw["month_change_pct"]
)


print(
    "Maximum absolute YoY difference:",
    rgia_traffic_raw["yoy_pct_difference"]
    .abs()
    .max()
)

print(
    "Records within 0.05 percentage points:",
    (
        rgia_traffic_raw["yoy_pct_difference"].abs() <= 0.05
    ).sum(),
    "out of",
    len(rgia_traffic_raw)
)

print("\nLargest differences:")

print(
    rgia_traffic_raw[
        [
            "report_month",
            "metric",
            "traffic_type",
            "month_change_pct",
            "calculated_month_change_pct",
            "yoy_pct_difference"
        ]
    ]
    .sort_values(
        "yoy_pct_difference",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(10)
)

Maximum absolute YoY difference: 0.04979923124033192
Records within 0.05 percentage points: 72 out of 72

Largest differences:
   report_month              metric   traffic_type  month_change_pct  \
24      2026-03          Passengers  International             -24.8   
21      2026-03             Freight  International              -8.9   
50      2026-06             Freight       Combined              24.2   
36      2026-05  Aircraft Movements  International              -8.9   
58      2026-07             Freight       Domestic              -4.9   
6       2026-01          Passengers  International              23.4   
44      2026-05          Passengers       Combined              -8.6   
15      2026-02          Passengers  International              14.2   
2       2026-01  Aircraft Movements       Combined              -5.1   
39      2026-05             Freight  International              32.9   

    calculated_month_change_pct  yoy_pct_difference  
24                   -24.7

In [47]:
# ============================================================
# Create Validated Base Dataset
# ============================================================

validated_columns = [
    "report_month",
    "metric",
    "traffic_type",
    "month_current",
    "month_previous_year",
    "month_change_pct",
    "period_current",
    "period_previous_year",
    "period_change_pct"
]

rgia_traffic_validated = (
    rgia_traffic_raw[validated_columns]
    .copy()
)

# Convert report_month to proper date
rgia_traffic_validated["report_month"] = pd.to_datetime(
    rgia_traffic_validated["report_month"],
    format="%Y-%m"
)

# Sort
rgia_traffic_validated = (
    rgia_traffic_validated
    .sort_values(
        [
            "report_month",
            "metric",
            "traffic_type"
        ]
    )
    .reset_index(drop=True)
)

# Save
VALIDATED_FILE = os.path.join(
    PROCESSED_DIR,
    "rgia_traffic_monthly_2026.csv"
)

rgia_traffic_validated.to_csv(
    VALIDATED_FILE,
    index=False
)

print("Validated dataset saved successfully.")
print("File:", VALIDATED_FILE)
print("Rows:", len(rgia_traffic_validated))
print("Columns:", len(rgia_traffic_validated.columns))

rgia_traffic_validated.head(12)

Validated dataset saved successfully.
File: /content/rgia_airport_analytics/data/processed/rgia_traffic_monthly_2026.csv
Rows: 72
Columns: 9


,report_month,metric,traffic_type,month_current,month_previous_year,month_change_pct,period_current,period_previous_year,period_change_pct
0,2026-01-01,Aircraft Movements,Combined,16894.0,17793.0,-5.1,177092.0,166865.0,6.1
1,2026-01-01,Aircraft Movements,Domestic,13999.0,15271.0,-8.3,149557.0,141750.0,5.5
2,2026-01-01,Aircraft Movements,International,2895.0,2522.0,14.8,27535.0,25115.0,9.6
3,2026-01-01,Freight,Combined,14908.4,12789.4,16.6,151152.0,138672.3,9.0
4,2026-01-01,Freight,Domestic,5596.8,5487.2,2.0,60536.5,57730.9,4.9
5,2026-01-01,Freight,International,9311.5,7302.2,27.5,90615.5,80941.4,12.0
6,2026-01-01,Passengers,Combined,2652779.0,2642587.0,0.4,25878498.0,24039935.0,7.6
7,2026-01-01,Passengers,Domestic,2129340.0,2218574.0,-4.0,21160774.0,20104882.0,5.3
8,2026-01-01,Passengers,International,523439.0,424013.0,23.4,4717724.0,3935053.0,19.9
9,2026-02-01,Aircraft Movements,Combined,14901.0,16913.0,-11.9,191993.0,183778.0,4.5


In [48]:
# ============================================================
# Final Dataset Integrity Check
# ============================================================

print("Shape:", rgia_traffic_validated.shape)

print("\nMissing values:")
print(
    rgia_traffic_validated.isna().sum()
)

print("\nDuplicate rows:")
print(
    rgia_traffic_validated.duplicated().sum()
)

print("\nUnique months:")
print(
    rgia_traffic_validated["report_month"]
    .dt.strftime("%Y-%m")
    .unique()
)

print("\nUnique metrics:")
print(
    rgia_traffic_validated["metric"].unique()
)

print("\nUnique traffic types:")
print(
    rgia_traffic_validated["traffic_type"].unique()
)

Shape: (72, 9)

Missing values:
report_month            0
metric                  0
traffic_type            0
month_current           0
month_previous_year     0
month_change_pct        0
period_current          0
period_previous_year    0
period_change_pct       0
dtype: int64

Duplicate rows:
0

Unique months:
['2026-01' '2026-02' '2026-03' '2026-04' '2026-05' '2026-06' '2026-07'
 '2026-08']

Unique metrics:
['Aircraft Movements' 'Freight' 'Passengers']

Unique traffic types:
['Combined' 'Domestic' 'International']


In [49]:
# ============================================================
# Operational KPI Dataset
# ============================================================

# ------------------------------------------------------------
# Extract monthly values by metric and traffic type
# ------------------------------------------------------------

passengers = rgia_traffic_validated[
    rgia_traffic_validated["metric"] == "Passengers"
].copy()

movements = rgia_traffic_validated[
    rgia_traffic_validated["metric"] == "Aircraft Movements"
].copy()

freight = rgia_traffic_validated[
    rgia_traffic_validated["metric"] == "Freight"
].copy()


# ------------------------------------------------------------
# Keep only the values needed for KPI construction
# ------------------------------------------------------------

passenger_pivot = passengers.pivot(
    index="report_month",
    columns="traffic_type",
    values="month_current"
).reset_index()

passenger_pivot = passenger_pivot.rename(
    columns={
        "International": "international_passengers",
        "Domestic": "domestic_passengers",
        "Combined": "total_passengers"
    }
)


movement_pivot = movements.pivot(
    index="report_month",
    columns="traffic_type",
    values="month_current"
).reset_index()

movement_pivot = movement_pivot.rename(
    columns={
        "International": "international_movements",
        "Domestic": "domestic_movements",
        "Combined": "total_movements"
    }
)


freight_pivot = freight.pivot(
    index="report_month",
    columns="traffic_type",
    values="month_current"
).reset_index()

freight_pivot = freight_pivot.rename(
    columns={
        "International": "international_freight_mt",
        "Domestic": "domestic_freight_mt",
        "Combined": "total_freight_mt"
    }
)


# ------------------------------------------------------------
# Combine datasets
# ------------------------------------------------------------

rgia_monthly_kpi = (
    passenger_pivot
    .merge(
        movement_pivot,
        on="report_month",
        how="inner"
    )
    .merge(
        freight_pivot,
        on="report_month",
        how="inner"
    )
)


# ------------------------------------------------------------
# Operational KPIs
# ------------------------------------------------------------

rgia_monthly_kpi["passengers_per_aircraft_movement"] = (
    rgia_monthly_kpi["total_passengers"]
    / rgia_monthly_kpi["total_movements"]
)

rgia_monthly_kpi["freight_per_aircraft_movement_mt"] = (
    rgia_monthly_kpi["total_freight_mt"]
    / rgia_monthly_kpi["total_movements"]
)


# ------------------------------------------------------------
# Passenger mix
# ------------------------------------------------------------

rgia_monthly_kpi["domestic_passenger_share_pct"] = (
    rgia_monthly_kpi["domestic_passengers"]
    / rgia_monthly_kpi["total_passengers"]
    * 100
)

rgia_monthly_kpi["international_passenger_share_pct"] = (
    rgia_monthly_kpi["international_passengers"]
    / rgia_monthly_kpi["total_passengers"]
    * 100
)


# ------------------------------------------------------------
# Aircraft movement mix
# ------------------------------------------------------------

rgia_monthly_kpi["domestic_movement_share_pct"] = (
    rgia_monthly_kpi["domestic_movements"]
    / rgia_monthly_kpi["total_movements"]
    * 100
)

rgia_monthly_kpi["international_movement_share_pct"] = (
    rgia_monthly_kpi["international_movements"]
    / rgia_monthly_kpi["total_movements"]
    * 100
)


# ------------------------------------------------------------
# Month-over-month passenger growth
# ------------------------------------------------------------

rgia_monthly_kpi["passenger_mom_growth_pct"] = (
    rgia_monthly_kpi["total_passengers"]
    .pct_change()
    * 100
)


# ------------------------------------------------------------
# Month-over-month movement growth
# ------------------------------------------------------------

rgia_monthly_kpi["movement_mom_growth_pct"] = (
    rgia_monthly_kpi["total_movements"]
    .pct_change()
    * 100
)


# ------------------------------------------------------------
# Month-over-month freight growth
# ------------------------------------------------------------

rgia_monthly_kpi["freight_mom_growth_pct"] = (
    rgia_monthly_kpi["total_freight_mt"]
    .pct_change()
    * 100
)


# ------------------------------------------------------------
# Final ordering
# ------------------------------------------------------------

rgia_monthly_kpi = (
    rgia_monthly_kpi
    .sort_values("report_month")
    .reset_index(drop=True)
)


print("KPI dataset shape:", rgia_monthly_kpi.shape)

rgia_monthly_kpi

KPI dataset shape: (8, 19)


traffic_type,report_month,total_passengers,domestic_passengers,international_passengers,total_movements,domestic_movements,international_movements,total_freight_mt,domestic_freight_mt,international_freight_mt,passengers_per_aircraft_movement,freight_per_aircraft_movement_mt,domestic_passenger_share_pct,international_passenger_share_pct,domestic_movement_share_pct,international_movement_share_pct,passenger_mom_growth_pct,movement_mom_growth_pct,freight_mom_growth_pct
0,2026-01-01,2652779.0,2129340.0,523439.0,16894.0,13999.0,2895.0,14908.4,5596.8,9311.5,157.024920,0.882467,80.268277,19.731723,82.863739,17.136261,NaN,NaN,NaN
1,2026-02-01,2317113.0,1887511.0,429602.0,14901.0,12328.0,2573.0,15447.9,5455.2,9992.7,155.500503,1.036702,81.459601,18.540399,82.732703,17.267297,-12.653372,-11.797088,3.618765
2,2026-03-01,2288274.0,1972498.0,315776.0,15907.0,13953.0,1954.0,15840.1,6172.9,9667.2,143.853272,0.995794,86.200254,13.799746,87.716100,12.283900,-1.244609,6.751225,2.538856
3,2026-04-01,2249965.0,1909069.0,340896.0,15829.0,13819.0,2010.0,16160.6,5843.6,10317.0,142.141955,1.020949,84.848831,15.151169,87.301788,12.698212,-1.674144,-0.490350,2.023346
4,2026-05-01,2549700.0,2135341.0,414359.0,17306.0,14915.0,2391.0,16549.3,6159.4,10390.0,147.330406,0.956275,83.748716,16.251284,86.183982,13.816018,13.321763,9.330975,2.405232
5,2026-06-01,2287844.0,1857680.0,430164.0,15356.0,12733.0,2623.0,17970.2,5888.1,12082.1,148.986976,1.170240,81.197844,18.802156,82.918729,17.081271,-10.270071,-11.267768,8.585862
6,2026-07-01,2223504.0,1743556.0,479948.0,14364.0,11614.0,2750.0,18070.8,5980.2,12090.6,154.796992,1.258062,78.414790,21.585210,80.854915,19.145085,-2.812255,-6.460016,0.559816
7,2026-08-01,2185934.0,1733770.0,452164.0,14812.0,12082.0,2730.0,17489.1,5848.5,11640.6,147.578585,1.180739,79.314838,20.685162,81.568998,18.431002,-1.689675,3.118908,-3.219005


In [50]:

# Final KPI Dataset — Integrity Check & Save

# Remove accidental index name if present
rgia_monthly_kpi.index.name = None

# Make sure there is no unwanted traffic_type column
print("Columns:")
print(rgia_monthly_kpi.columns.tolist())

print("\nShape:", rgia_monthly_kpi.shape)

print("\nMissing values:")
print(rgia_monthly_kpi.isna().sum())

print("\nDuplicate rows:")
print(rgia_monthly_kpi.duplicated().sum())

Columns:
['report_month', 'total_passengers', 'domestic_passengers', 'international_passengers', 'total_movements', 'domestic_movements', 'international_movements', 'total_freight_mt', 'domestic_freight_mt', 'international_freight_mt', 'passengers_per_aircraft_movement', 'freight_per_aircraft_movement_mt', 'domestic_passenger_share_pct', 'international_passenger_share_pct', 'domestic_movement_share_pct', 'international_movement_share_pct', 'passenger_mom_growth_pct', 'movement_mom_growth_pct', 'freight_mom_growth_pct']

Shape: (8, 19)

Missing values:
traffic_type
report_month                         0
total_passengers                     0
domestic_passengers                  0
international_passengers             0
total_movements                      0
domestic_movements                   0
international_movements              0
total_freight_mt                     0
domestic_freight_mt                  0
international_freight_mt             0
passengers_per_aircraft_movement     0


In [51]:

# Save Operational KPI Dataset

KPI_FILE = os.path.join(
    PROCESSED_DIR,
    "rgia_monthly_operational_kpis_2026.csv"
)

rgia_monthly_kpi.to_csv(
    KPI_FILE,
    index=False
)

print("KPI dataset saved successfully.")
print("File:", KPI_FILE)

KPI dataset saved successfully.
File: /content/rgia_airport_analytics/data/processed/rgia_monthly_operational_kpis_2026.csv


In [52]:
from google.colab import files

files.download(
    "/content/rgia_airport_analytics/data/processed/rgia_traffic_monthly_2026.csv"
)

files.download(
    "/content/rgia_airport_analytics/data/processed/rgia_monthly_operational_kpis_2026.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>